# RevalExo external window adapter

Create model-shaped 5-second, 100-Hz windows from HC/ST level-ground walking only. These outputs are external-validation data and must not enter training or normalization.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
ROOT=Path('../data/raw/revalexo').resolve(); OUT=Path('../data/processed')
FS_OUT=100.; WIN=500; HOP=250; G=9.80665; RAD2DEG=180/np.pi
rows=[]; windows=[]
for folder in sorted(ROOT.glob('raw_full_part*/raw_full/Subject*')):
    group=folder.name.rsplit('_',1)[-1]
    if group not in {'HC','ST'}: continue
    ann=pd.read_csv(folder/'annotations.csv'); ann=ann[ann['Task'].str.contains('level ground walking',case=False,na=False)]
    if ann.empty: continue
    with h5py.File(folder/'mvn-analyze.hdf5','r') as f:
        g=f['mvn-analyze/xsens-motion-trackers']; t=g['process_time_s'][:].ravel(); fs=1/np.median(np.diff(t))
        acc=np.asarray(g['free_acceleration'][:,[0,3,6],:],dtype=np.float32)/G
        gyro=np.asarray(g['gyroscope'][:,[0,3,6],:],dtype=np.float32)*RAD2DEG
        signal=np.concatenate([acc[:,0],gyro[:,0],acc[:,2],gyro[:,2],acc[:,1],gyro[:,1]],axis=1)
        target=np.arange(t[0],t[-1],1/FS_OUT)
        res=np.column_stack([np.interp(target,t,signal[:,j]) for j in range(18)]).astype(np.float32)
    for _,w in ann.iterrows():
        start,end=float(w['Start_toa_s']),float(w['End_toa_s'])
        a=max(0,int(np.searchsorted(target,start))); b=min(len(target),int(np.searchsorted(target,end)))
        for s in range(a,b-WIN+1,HOP):
            wid=len(windows); windows.append(res[s:s+WIN]); rows.append({'window_id':wid,'subject':folder.name,'group':group,'start_time_s':start,'window_start_index':s,'window_seconds':5.0,'source_hz':fs})
meta=pd.DataFrame(rows); arr=np.asarray(windows,dtype=np.float32)
assert arr.ndim==3 and arr.shape[1:]==(500,18) and np.isfinite(arr).all()
np.save(OUT/'revalexo_external_windows_float32.npy',arr); meta.to_csv(OUT/'revalexo_external_window_metadata.csv',index=False)
print('Windows:',arr.shape); print(meta.groupby('group').agg(subjects=('subject','nunique'),windows=('window_id','size')))

Windows: (2228, 500, 18)
       subjects  windows
group                   
HC            7      754
ST           10     1474
